In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('..'))

import cv2
import time
import glob
import numpy as np
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from models.UNet import BaseUNet, AttentionUNet
from models.ResNet18_UNet import ResNet18_UNet
from models.MobileNetV2_UNet import MobileNetV2_UNet

# Hyper-parameters

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 8
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_RUNS = 5
EPOCHS = 80
LR = 1e-4
WEIGHT_DECAY = 5e-4

torch.backends.cudnn.benchmark = True
print(f"Using device: {DEVICE}")

# Load BUSI Dataset

In [ ]:
path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
print(f"Dataset path: {path}")

busi_root = None
for root, dirs, files in os.walk(path):
    if 'Dataset_BUSI_with_GT' in dirs:
        busi_root = os.path.join(root, 'Dataset_BUSI_with_GT')
        break
    if os.path.basename(root) == 'Dataset_BUSI_with_GT':
        busi_root = root
        break
if busi_root is None:
    candidates = glob.glob(os.path.join(path, '**', 'benign'), recursive=True)
    if candidates: busi_root = os.path.dirname(candidates[0])
assert busi_root is not None, f"Could not find BUSI dataset under {path}"
print(f"BUSI root: {busi_root}")

entries = []
for category in ['benign', 'malignant']:
    cat_dir = os.path.join(busi_root, category)
    if not os.path.isdir(cat_dir): continue
    files = sorted(os.listdir(cat_dir))
    image_files = [f for f in files if '_mask' not in f and f.endswith('.png')]
    for img_name in image_files:
        img_path = os.path.join(cat_dir, img_name)
        base = img_name.replace('.png', '')
        mask_path = os.path.join(cat_dir, f"{base}_mask.png")
        if os.path.exists(mask_path):
            entries.append((category, img_path, mask_path))

df = pd.DataFrame(entries, columns=['label', 'image_path', 'mask_path'])
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'])
print(f"Found {len(df)} BUSI pairs. Train: {len(train_df)}, Val: {len(val_df)}")

In [ ]:
class BUSIDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = cv2.imread(self.df.loc[idx, 'image_path'])
        if img is None:
            img = np.zeros((*IMG_SIZE, 3), np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.df.loc[idx, 'mask_path'], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros(IMG_SIZE, np.uint8)
        mask = (mask > 0).astype('uint8')
        aug = self.transform(image=img, mask=mask)
        return aug['image'], aug['mask'].unsqueeze(0).float()

In [ ]:
train_transform = A.Compose([
    A.Resize(*IMG_SIZE), A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(0.05, 0.1, 15, p=0.5),
    A.ElasticTransform(alpha=50, sigma=50*0.05, p=0.3),
    A.RandomBrightnessContrast(p=0.8), A.GaussNoise(p=0.3),
    A.Normalize(mean=(0.0,0.0,0.0), std=(1.0,1.0,1.0), max_pixel_value=255.0),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(*IMG_SIZE),
    A.Normalize(mean=(0.0,0.0,0.0), std=(1.0,1.0,1.0), max_pixel_value=255.0),
    ToTensorV2()
])

train_loader = DataLoader(BUSIDataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(BUSIDataset(val_df, val_transform),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
print("BUSI data ready.")

# Loss and Metrics

In [ ]:
from utils.metric import BCEDiceLoss

MIOU_THRESHOLDS = np.arange(0.5, 0.96, 0.05)

def compute_metrics(logits, targets):
    probs = torch.sigmoid(logits)
    bp = (probs > 0.5).float()
    tp = (bp * targets).sum((1,2,3))
    fp = (bp * (1-targets)).sum((1,2,3))
    fn = ((1-bp) * targets).sum((1,2,3))
    dice = (2*tp / (2*tp + fp + fn + 1e-6)).mean().item()
    prec = (tp / (tp + fp + 1e-6)).mean().item()
    ious = []
    for t in MIOU_THRESHOLDS:
        bp_t = (probs > t).float()
        tp_t = (bp_t * targets).sum((1,2,3))
        fp_t = (bp_t * (1-targets)).sum((1,2,3))
        fn_t = ((1-bp_t) * targets).sum((1,2,3))
        ious.append((tp_t / (tp_t + fp_t + fn_t + 1e-6)).mean().item())
    return dice, prec, np.mean(ious)

# Run All 4 Models on BUSI

Uncomment one at a time to avoid Colab disconnects.

In [ ]:
def run_experiment(name, model_fn, num_runs=NUM_RUNS, epochs=EPOCHS):
    print(f"\n{'='*60}")
    print(f"  BUSI EXPERIMENT: {name}")
    print(f"  {num_runs} runs x {epochs} epochs")
    print(f"{'='*60}")
    all_dice, all_prec, all_miou = [], [], []
    for run in range(1, num_runs+1):
        print(f"\n--- Run {run}/{num_runs} ---")
        model = model_fn().to(DEVICE)
        criterion = BCEDiceLoss(weight_bce=0.5, weight_dice=0.5)
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)
        best_dice, best_prec, best_miou = 0, 0, 0
        for ep in range(1, epochs+1):
            model.train()
            for imgs, msks in train_loader:
                imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(imgs), msks)
                loss.backward(); optimizer.step()
            model.eval()
            ep_dice, ep_prec, ep_miou = [], [], []
            with torch.no_grad():
                for imgs, msks in val_loader:
                    imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
                    d, p, m = compute_metrics(model(imgs), msks)
                    ep_dice.append(d); ep_prec.append(p); ep_miou.append(m)
            vd, vp, vm = np.mean(ep_dice), np.mean(ep_prec), np.mean(ep_miou)
            scheduler.step(vm)
            if vm > best_miou:
                best_dice, best_prec, best_miou = vd, vp, vm
            if ep % 20 == 0 or ep == 1:
                print(f"  Ep {ep:3d}/{epochs}  Dice:{vd:.4f}  mIoU:{vm:.4f}")
        all_dice.append(best_dice)
        all_prec.append(best_prec)
        all_miou.append(best_miou)
        print(f"  Run {run} best -> Dice:{best_dice:.4f} Prec:{best_prec:.4f} mIoU:{best_miou:.4f}")
    da, pa, ma = np.array(all_dice), np.array(all_prec), np.array(all_miou)
    print(f"\n  {name} BUSI SUMMARY:")
    print(f"  Dice : {da.mean()*100:.2f} +/- {da.std()*100:.2f} %")
    print(f"  Prec : {pa.mean()*100:.2f} +/- {pa.std()*100:.2f} %")
    print(f"  mIoU : {ma.mean()*100:.2f} +/- {ma.std()*100:.2f} %")
    return {'name': name, 'dice': da.mean()*100, 'dice_std': da.std()*100,
            'prec': pa.mean()*100, 'prec_std': pa.std()*100,
            'miou': ma.mean()*100, 'miou_std': ma.std()*100}

In [ ]:
results = []

# Experiment 1: Base U-Net
results.append(run_experiment("Base U-Net", lambda: BaseUNet()))

# Experiment 2: Attention U-Net
# results.append(run_experiment("Attention U-Net", lambda: AttentionUNet()))

# Experiment 3: ResNet18-UNet
# results.append(run_experiment("ResNet18-UNet", lambda: ResNet18_UNet()))

# Experiment 4: Proposed MobileNetV2-UNet
# results.append(run_experiment("MobileNetV2-UNet", lambda: MobileNetV2_UNet()))

# Summary

In [ ]:
print("\n" + "="*70)
print("  BUSI RESULTS")
print("="*70)
for r in results:
    print(f"  {r['name']:<25s}  Dice: {r['dice']:.2f}+/-{r['dice_std']:.2f}  "
          f"Prec: {r['prec']:.2f}+/-{r['prec_std']:.2f}  "
          f"mIoU: {r['miou']:.2f}+/-{r['miou_std']:.2f}")
print("="*70)